# Análise de sinistros nas rodovias federais do Brasil

Este notebook reúne a lógica do projeto em uma sequência única e documentada. Cada etapa explica o que faz, quais arquivos utiliza e quais resultados produz.

## Fluxo do processamento

1. Baixar os dados públicos da Polícia Rodoviária Federal.
2. Consolidar os arquivos anuais de ocorrência e pessoa/veículo.
3. Normalizar marca, modelo e família dos veículos.
4. Enriquecer os registros com resultados do Latin NCAP.
5. Enriquecer os registros com a base local da FIPE.
6. Filtrar veículos leves com pareamento Latin NCAP.
7. Consolidar a base analítica e gerar o resumo de cobertura.

A célula final executa o fluxo completo com retomada e gera o CSV analítico consolidado.


## Preparação do ambiente

As funções abaixo carregam os módulos incorporados nas próximas seções. O código-fonte de cada script está armazenado no próprio notebook para que ele possa ser lido e executado sem depender da chamada de outros scripts por `subprocess`.


In [ ]:
from pathlib import Path
import sys
import types

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "scripts").exists() and (RAIZ_PROJETO.parent / "scripts").exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

if str(RAIZ_PROJETO / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO / "scripts"))

modulos = {}
print(f"Pasta do projeto: {RAIZ_PROJETO}")


### download_prf_data.py

Baixa e extrai os arquivos anuais dos dados abertos da PRF, registrando um manifesto.

In [ ]:
modulo_download_prf_data = types.ModuleType('notebook_download_prf_data')
sys.modules['notebook_download_prf_data'] = modulo_download_prf_data
namespace_download_prf_data = modulo_download_prf_data.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
import html
import re
import subprocess
import sys
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


PRF_OPEN_DATA_URL = (
    "https://www.gov.br/prf/pt-br/acesso-a-informacao/"
    "dados-abertos/dados-abertos-da-prf"
)
USER_AGENT = "prf-analise-downloader/1.0"
DEFAULT_KINDS = ("ocorrencia", "pessoa")
MANIFEST_FIELDS = [
    "year",
    "kind",
    "label",
    "source_url",
    "file_id",
    "download_url",
    "local_zip",
    "extract_dir",
    "extracted_files",
    "status",
    "bytes",
]


@dataclass(frozen=True)
class PrfDownload:
    year: int
    kind: str
    label: str
    source_url: str
    file_id: str

    @property
    def download_url(self) -> str:
        return f"https://drive.google.com/uc?export=download&id={self.file_id}"


def request_url(url: str, timeout: int = 90) -> urllib.request.Request:
    return urllib.request.Request(url, headers={"User-Agent": USER_AGENT})


def fetch_html(url: str) -> str:
    with urllib.request.urlopen(request_url(url), timeout=90) as response:
        return response.read().decode("utf-8", errors="replace")


def strip_tags(value: str) -> str:
    value = re.sub(r"<[^>]+>", " ", value)
    return " ".join(html.unescape(value).replace("\xa0", " ").split())


def drive_file_id(url: str) -> str:
    match = re.search(r"/file/d/([^/]+)/", url)
    if match:
        return match.group(1)
    parsed = urllib.parse.urlparse(url)
    query = urllib.parse.parse_qs(parsed.query)
    return query.get("id", [""])[0]


def row_kind(label: str) -> str:
    normalized = label.lower()
    if "todas as causas" in normalized:
        return "pessoa_todas_causas"
    if "agrupados por ocorrência" in normalized or "agrupados por ocorrencia" in normalized:
        return "ocorrencia"
    if "agrupados por pessoa" in normalized:
        return "pessoa"
    return ""


def visible_download_href(cell_html: str) -> str:
    anchors = re.findall(
        r"<a\b(?P<attrs>[^>]*)>(?P<body>.*?)</a>",
        cell_html,
        flags=re.IGNORECASE | re.DOTALL,
    )
    fallback = ""
    for attrs, body in anchors:
        href_match = re.search(r'href="([^"]+)"', attrs)
        if not href_match:
            continue
        href = html.unescape(href_match.group(1))
        fallback = fallback or href
        if "baixar" in strip_tags(body).lower():
            return href
    return fallback


def parse_accident_links(page_html: str) -> list[PrfDownload]:
    rows: list[PrfDownload] = []
    for match in re.finditer(
        r"<tr>\s*<td>(?P<label>Documento CSV de Acidentes .*?)</td>\s*"
        r"<td>(?P<link_cell>.*?)</td>\s*</tr>",
        page_html,
        flags=re.IGNORECASE | re.DOTALL,
    ):
        label = strip_tags(match.group("label"))
        year_match = re.search(r"\b(20\d{2})\b", label)
        kind = row_kind(label)
        href = visible_download_href(match.group("link_cell"))
        file_id = drive_file_id(href)
        if not year_match or not kind or not href or not file_id:
            continue
        rows.append(
            PrfDownload(
                year=int(year_match.group(1)),
                kind=kind,
                label=label,
                source_url=href,
                file_id=file_id,
            )
        )
    rows.sort(key=lambda row: (row.year, row.kind))
    return rows


def selected_links(
    links: Iterable[PrfDownload],
    start_year: int,
    end_year: int,
    kinds: set[str],
) -> list[PrfDownload]:
    selected = [
        link
        for link in links
        if start_year <= link.year <= end_year and link.kind in kinds
    ]
    selected.sort(key=lambda row: (row.year, row.kind))
    return selected


def download_file(url: str, output_path: Path, overwrite: bool) -> int:
    if output_path.exists() and not overwrite:
        return output_path.stat().st_size

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")
    if temporary_path.exists():
        temporary_path.unlink()
    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            "--connect-timeout",
            "30",
            "--max-time",
            "300",
            "--silent",
            "--show-error",
            "-o",
            str(temporary_path),
            url,
        ],
        check=True,
    )
    temporary_path.replace(output_path)
    return output_path.stat().st_size


def extract_zip(zip_path: Path, extract_dir: Path, overwrite: bool) -> list[str]:
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        names = [name for name in archive.namelist() if not name.endswith("/")]
        for name in names:
            target = extract_dir / name
            if target.exists() and not overwrite:
                continue
            archive.extract(name, extract_dir)
    return names


def write_manifest(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS)
        writer.writeheader()
        writer.writerows(rows)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--source-url", default=PRF_OPEN_DATA_URL)
    parser.add_argument("--start-year", type=int, default=2010)
    parser.add_argument("--end-year", type=int, default=2026)
    parser.add_argument(
        "--kinds",
        nargs="+",
        choices=("ocorrencia", "pessoa", "pessoa_todas_causas"),
        default=list(DEFAULT_KINDS),
    )
    parser.add_argument("--output-dir", type=Path, default=Path("data/raw/prf/acidentes"))
    parser.add_argument(
        "--manifest-output",
        type=Path,
        default=Path("data/raw/prf/manifest_acidentes_2010_2026.csv"),
    )
    parser.add_argument("--dry-run", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--limit", type=int, default=0)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    links = selected_links(
        parse_accident_links(fetch_html(args.source_url)),
        args.start_year,
        args.end_year,
        set(args.kinds),
    )
    if args.limit:
        links = links[: args.limit]

    manifest_rows: list[dict[str, object]] = []
    for position, link in enumerate(links, start=1):
        zip_path = args.output_dir / link.kind / f"{link.year}.zip"
        extract_dir = args.output_dir / link.kind / str(link.year)
        row = {
            "year": link.year,
            "kind": link.kind,
            "label": link.label,
            "source_url": link.source_url,
            "file_id": link.file_id,
            "download_url": link.download_url,
            "local_zip": str(zip_path),
            "extract_dir": str(extract_dir),
            "extracted_files": "",
            "status": "dry_run" if args.dry_run else "pending",
            "bytes": "",
        }
        if not args.dry_run:
            try:
                print(
                    f"[{position}/{len(links)}] Baixando {link.year} {link.kind}",
                    flush=True,
                )
                size = download_file(link.download_url, zip_path, overwrite=args.overwrite)
                extracted = extract_zip(zip_path, extract_dir, overwrite=args.overwrite)
                row.update(
                    {
                        "extracted_files": "|".join(extracted),
                        "status": "ok",
                        "bytes": size,
                    }
                )
            except (
                OSError,
                subprocess.CalledProcessError,
                zipfile.BadZipFile,
                urllib.error.URLError,
            ) as exc:
                row["status"] = f"erro: {exc}"
                print(f"Erro em {link.year} {link.kind}: {exc}", file=sys.stderr)
        manifest_rows.append(row)

    write_manifest(args.manifest_output, manifest_rows)
    print(f"Links selecionados: {len(links)}")
    print(f"Manifesto: {args.manifest_output}")


''', 'download_prf_data.py', 'exec'), namespace_download_prf_data)
modulos['download_prf_data'] = namespace_download_prf_data
print('Código incorporado: download_prf_data.py')


### build_prf_multiyear.py

Consolida arquivos CSV de vários anos, preservando a origem e unificando as colunas.

In [ ]:
modulo_build_prf_multiyear = types.ModuleType('notebook_build_prf_multiyear')
sys.modules['notebook_build_prf_multiyear'] = modulo_build_prf_multiyear
namespace_build_prf_multiyear = modulo_build_prf_multiyear.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
from pathlib import Path
from typing import Iterable


SOURCE_EXTRA_FIELDS = ["ano_arquivo", "arquivo_origem"]


def csv_paths(input_dir: Path) -> list[Path]:
    paths = sorted(input_dir.glob("*/*.csv"))
    if not paths:
        paths = sorted(input_dir.glob("**/*.csv"))
    return paths


def open_with_known_encoding(path: Path):
    for encoding in ("utf-8-sig", "latin1"):
        try:
            handle = path.open("r", encoding=encoding, newline="")
            handle.read(4096)
            handle.seek(0)
            return handle
        except UnicodeDecodeError:
            handle.close()
    return path.open("r", encoding="latin1", newline="")


def detect_delimiter(path: Path) -> str:
    with open_with_known_encoding(path) as handle:
        first_line = handle.readline()
    return ";" if first_line.count(";") >= first_line.count(",") else ","


def read_header(path: Path) -> list[str]:
    delimiter = detect_delimiter(path)
    with open_with_known_encoding(path) as handle:
        return csv.DictReader(handle, delimiter=delimiter).fieldnames or []


def union_fieldnames(
    paths: Iterable[Path], source_extra_fields: Iterable[str] = SOURCE_EXTRA_FIELDS
) -> list[str]:
    fields: list[str] = []
    seen = set()
    for extra_field in source_extra_fields:
        fields.append(extra_field)
        seen.add(extra_field)
    for path in paths:
        for field in read_header(path):
            if field not in seen:
                fields.append(field)
                seen.add(field)
    return fields


def year_from_path(path: Path) -> str:
    for part in reversed(path.parts):
        if part.isdigit() and len(part) == 4:
            return part
    digits = "".join(character for character in path.stem if character.isdigit())
    return digits[:4]


def combine(
    input_dir: Path,
    output_path: Path,
    output_delimiter: str = ";",
    source_extra_fields: Iterable[str] = SOURCE_EXTRA_FIELDS,
) -> int:
    paths = csv_paths(input_dir)
    if not paths:
        raise FileNotFoundError(f"Nenhum CSV encontrado em {input_dir}")

    source_extra_fields = list(source_extra_fields)
    fieldnames = union_fieldnames(paths, source_extra_fields)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with output_path.open("w", encoding="utf-8", newline="") as output_handle:
        writer = csv.DictWriter(
            output_handle,
            fieldnames=fieldnames,
            delimiter=output_delimiter,
            extrasaction="ignore",
        )
        writer.writeheader()
        for path in paths:
            input_delimiter = detect_delimiter(path)
            with open_with_known_encoding(path) as input_handle:
                reader = csv.DictReader(input_handle, delimiter=input_delimiter)
                for row in reader:
                    source_fields = {}
                    if source_extra_fields:
                        source_fields = {
                            "ano_arquivo": year_from_path(path),
                            "arquivo_origem": str(path),
                        }
                    writer.writerow({**row, **source_fields})
                    count += 1
    return count


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--pessoa-dir", type=Path, required=True)
    parser.add_argument("--ocorrencia-dir", type=Path, required=True)
    parser.add_argument("--pessoa-output", type=Path, required=True)
    parser.add_argument("--ocorrencia-output", type=Path, required=True)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    pessoa_rows = combine(args.pessoa_dir, args.pessoa_output)
    ocorrencia_rows = combine(args.ocorrencia_dir, args.ocorrencia_output)
    print(f"Registros por pessoa: {pessoa_rows} em {args.pessoa_output}")
    print(f"Registros por ocorrencia: {ocorrencia_rows} em {args.ocorrencia_output}")


''', 'build_prf_multiyear.py', 'exec'), namespace_build_prf_multiyear)
modulos['build_prf_multiyear'] = namespace_build_prf_multiyear
print('Código incorporado: build_prf_multiyear.py')


### enrich_fipe.py

Contém normalização de textos, classificação de categorias e a integração original com a API FIPE.

In [ ]:
modulo_enrich_fipe = types.ModuleType('notebook_enrich_fipe')
sys.modules['notebook_enrich_fipe'] = modulo_enrich_fipe
namespace_enrich_fipe = modulo_enrich_fipe.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import re
import statistics
import time
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Iterable
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


DEFAULT_API_BASE_URL = "https://parallelum.com.br/fipe/api/v1"
INVALID_VALUES = {"", "NA", "NA NA", "NAO INFORMADO NAO INFORMADO"}

BRAND_ALIASES = {
    "CHEV": "CHEVROLET",
    "GM": "CHEVROLET",
    "M BENZ": "MERCEDES BENZ",
    "MBENZ": "MERCEDES BENZ",
    "M BENS": "MERCEDES BENZ",
    "MMC": "MITSUBISHI",
    "VW": "VOLKSWAGEN",
}

CATEGORY_BY_PRF_TYPE = {
    "AUTOMOVEL": "carros",
    "CAMINHONETE": "carros",
    "CAMIONETA": "carros",
    "UTILITARIO": "carros",
    "MOTOCICLETA": "motos",
    "MOTONETA": "motos",
    "CICLOMOTOR": "motos",
    "CAMINHAO": "caminhoes",
    "CAMINHAO TRATOR": "caminhoes",
    "ONIBUS": "caminhoes",
    "MICRO ONIBUS": "caminhoes",
}

EXTRACT_FIELDS = [
    "id_veiculo",
    "tipo_veiculo",
    "marca_modelo_original",
    "ano_fabricacao_veiculo",
    "categoria_fipe",
    "marca_prf",
    "modelo_prf",
    "marca_normalizada",
    "modelo_normalizado",
    "quantidade_registros_envolvidos",
    "quantidade_acidentes",
]

MATCH_FIELDS = [
    "assinatura_veiculo",
    "categoria_fipe",
    "marca_modelo_original",
    "ano_fabricacao_veiculo",
    "marca_normalizada",
    "modelo_normalizado",
    "quantidade_veiculos",
    "status_match",
    "confianca_match",
    "score_match",
    "margem_segundo_candidato",
    "quantidade_candidatos_plausiveis",
    "marca_fipe",
    "modelo_fipe_sugerido",
    "ano_modelo_fipe_sugerido",
    "combustivel_fipe_sugerido",
    "codigo_fipe_sugerido",
    "valor_fipe_sugerido",
    "valor_fipe_min",
    "valor_fipe_max",
    "valor_fipe_mediano",
    "mes_referencia_fipe",
    "metodo_match",
    "observacao",
]


def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFKD", value or "")
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r"(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])", " ", value)
    value = re.sub(r"[^A-Za-z0-9]+", " ", value.upper())
    return re.sub(r"\s+", " ", value).strip()


def normalize_brand(value: str) -> str:
    normalized = normalize_text(value)
    return BRAND_ALIASES.get(normalized, normalized)


def split_brand_model(value: str) -> tuple[str, str]:
    parts = [part.strip() for part in (value or "").split("/")]
    if len(parts) < 2:
        return (parts[0] if parts else "", "")
    if normalize_text(parts[0]) in {"I", "IMP", "IMPORTADO"}:
        imported_description = "/".join(parts[1:])
        imported_parts = imported_description.split(maxsplit=1)
        return imported_parts[0], imported_parts[1] if len(imported_parts) > 1 else ""
    return parts[0], "/".join(parts[1:])


def fipe_category(prf_vehicle_type: str) -> str:
    return CATEGORY_BY_PRF_TYPE.get(normalize_text(prf_vehicle_type), "")


def signature(row: dict[str, str]) -> str:
    values = (
        row.get("categoria_fipe", ""),
        row.get("marca_normalizada", ""),
        row.get("modelo_normalizado", ""),
        row.get("ano_fabricacao_veiculo", ""),
    )
    return "|".join(values)


def is_eligible_signature(row: dict[str, str]) -> bool:
    return bool(
        row.get("categoria_fipe")
        and normalize_text(row.get("marca_modelo_original", "")) not in INVALID_VALUES
        and parse_year(row.get("ano_fabricacao_veiculo", ""))
    )


def parse_year(value: str) -> int | None:
    try:
        year = int(value)
    except (TypeError, ValueError):
        return None
    return year if 1900 <= year <= 2100 else None


def parse_brl(value: str) -> int | None:
    normalized = re.sub(r"[^0-9,]", "", value or "").replace(",", ".")
    return round(float(normalized)) if normalized else None


def write_csv(path: Path, rows: Iterable[dict[str, Any]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def extract_unique_vehicles(input_path: Path, output_path: Path) -> None:
    vehicles: dict[str, dict[str, Any]] = {}
    with input_path.open("r", encoding="latin1", newline="") as handle:
        for row in csv.DictReader(handle, delimiter=";"):
            vehicle_id = row.get("id_veiculo", "")
            if vehicle_id in {"", "NA"}:
                continue
            current = vehicles.setdefault(
                vehicle_id,
                {
                    "id_veiculo": vehicle_id,
                    "tipo_veiculo": row.get("tipo_veiculo", ""),
                    "marca_modelo_original": row.get("marca", ""),
                    "ano_fabricacao_veiculo": row.get("ano_fabricacao_veiculo", ""),
                    "accident_ids": set(),
                    "quantidade_registros_envolvidos": 0,
                },
            )
            current["quantidade_registros_envolvidos"] += 1
            current["accident_ids"].add(row.get("id", ""))

    output_rows = []
    for vehicle in vehicles.values():
        brand, model = split_brand_model(vehicle["marca_modelo_original"])
        output_rows.append(
            {
                **vehicle,
                "categoria_fipe": fipe_category(vehicle["tipo_veiculo"]),
                "marca_prf": brand,
                "modelo_prf": model,
                "marca_normalizada": normalize_brand(brand),
                "modelo_normalizado": normalize_text(model),
                "quantidade_acidentes": len(vehicle["accident_ids"]),
            }
        )
    output_rows.sort(key=lambda row: int(row["id_veiculo"]))
    write_csv(output_path, output_rows, EXTRACT_FIELDS)
    print(f"Veiculos unicos gravados: {len(output_rows)} em {output_path}")


class FipeClient:
    def __init__(self, base_url: str, cache_dir: Path, delay_seconds: float = 0.05):
        self.base_url = base_url.rstrip("/")
        self.cache_dir = cache_dir
        self.delay_seconds = delay_seconds

    def get(self, path: str) -> Any:
        url = f"{self.base_url}/{path.lstrip('/')}"
        digest = hashlib.sha256(url.encode("utf-8")).hexdigest()
        cache_path = self.cache_dir / f"{digest}.json"
        if cache_path.exists():
            return json.loads(cache_path.read_text(encoding="utf-8"))

        request = Request(url, headers={"User-Agent": "prf-analise-fipe-enrichment/1.0"})
        try:
            with urlopen(request, timeout=30) as response:
                data = json.load(response)
        except (HTTPError, URLError, TimeoutError) as error:
            raise RuntimeError(f"Falha consultando {url}: {error}") from error

        self.cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        time.sleep(self.delay_seconds)
        return data


def resolve_fipe_brand(client: FipeClient, category: str, normalized_brand: str) -> dict[str, Any] | None:
    brands = client.get(f"{category}/marcas")
    exact_matches = [
        brand for brand in brands if normalize_brand(brand["nome"]) == normalized_brand
    ]
    if exact_matches:
        return exact_matches[0]

    ranked = sorted(
        (
            (
                SequenceMatcher(
                    None, normalized_brand, normalize_brand(brand["nome"])
                ).ratio(),
                brand,
            )
            for brand in brands
        ),
        key=lambda candidate: candidate[0],
    )
    if ranked and ranked[-1][0] >= 0.82:
        return ranked[-1][1]
    return None


def token_overlap(source: str, target: str) -> float:
    source_tokens = set(normalize_text(source).split())
    target_tokens = set(normalize_text(target).split())
    if not source_tokens:
        return 0.0
    return len(source_tokens & target_tokens) / len(source_tokens)


def model_similarity(source: str, target: str) -> float:
    source_normalized = normalize_text(source)
    target_normalized = normalize_text(target)
    sequence_ratio = SequenceMatcher(None, source_normalized, target_normalized).ratio()
    overlap = token_overlap(source_normalized, target_normalized)
    prefix_bonus = 0.10 if target_normalized.startswith(source_normalized) else 0.0
    return min(1.0, (0.55 * overlap) + (0.45 * sequence_ratio) + prefix_bonus)


def available_matching_years(years: list[dict[str, str]], fabrication_year: int | None) -> list[dict[str, str]]:
    if not fabrication_year:
        return []
    result = []
    for year in years:
        year_model = parse_year(year.get("nome", "").split()[0])
        if year_model in {fabrication_year, fabrication_year + 1}:
            result.append(year)
    return result


def price_candidate(
    client: FipeClient,
    category: str,
    brand: dict[str, Any],
    model: dict[str, Any],
    year: dict[str, str],
    similarity: float,
    fabrication_year: int,
    source_model: str,
) -> dict[str, Any]:
    detail = client.get(
        f"{category}/marcas/{brand['codigo']}/modelos/{model['codigo']}/anos/{year['codigo']}"
    )
    model_year = int(detail["AnoModelo"])
    year_points = 15 if model_year == fabrication_year else 10
    return {
        "score": 30 + round(similarity * 50) + year_points,
        "similarity": similarity,
        "token_overlap": token_overlap(source_model, model["nome"]),
        "modelo_codigo_api": model["codigo"],
        "marca_fipe": detail.get("Marca", brand["nome"]),
        "modelo_fipe": detail.get("Modelo", model["nome"]),
        "ano_modelo": model_year,
        "combustivel": detail.get("Combustivel", ""),
        "codigo_fipe": detail.get("CodigoFipe", ""),
        "valor": detail.get("Valor", ""),
        "valor_numero": parse_brl(detail.get("Valor", "")),
        "mes_referencia": detail.get("MesReferencia", ""),
    }


def confidence(score: int, margin: int) -> str:
    if score >= 90 and margin >= 8:
        return "alta"
    if score >= 65 and margin >= 5:
        return "media"
    if score >= 45:
        return "baixa"
    return "sem_match"


def unmatched_result(row: dict[str, str], quantity: int, status: str, observation: str) -> dict[str, Any]:
    return {
        "assinatura_veiculo": signature(row),
        "categoria_fipe": row.get("categoria_fipe", ""),
        "marca_modelo_original": row.get("marca_modelo_original", ""),
        "ano_fabricacao_veiculo": row.get("ano_fabricacao_veiculo", ""),
        "marca_normalizada": row.get("marca_normalizada", ""),
        "modelo_normalizado": row.get("modelo_normalizado", ""),
        "quantidade_veiculos": quantity,
        "status_match": status,
        "confianca_match": "sem_match",
        "metodo_match": "marca_modelo_ano_aproximado",
        "observacao": observation,
    }


def match_signature(
    client: FipeClient,
    row: dict[str, str],
    quantity: int,
    max_model_candidates: int,
    max_model_search: int,
) -> dict[str, Any]:
    category = row["categoria_fipe"]
    fabrication_year = parse_year(row["ano_fabricacao_veiculo"])
    if not category:
        return unmatched_result(row, quantity, "categoria_nao_suportada", "Tipo de veiculo fora das categorias FIPE usadas.")
    if normalize_text(row["marca_modelo_original"]) in INVALID_VALUES:
        return unmatched_result(row, quantity, "dados_insuficientes", "Marca/modelo ausente ou nao informado.")
    if not fabrication_year:
        return unmatched_result(row, quantity, "dados_insuficientes", "Ano de fabricacao ausente ou invalido.")

    brand = resolve_fipe_brand(client, category, row["marca_normalizada"])
    if not brand:
        return unmatched_result(row, quantity, "marca_sem_match", "Marca PRF nao associada a uma marca FIPE.")

    models = client.get(f"{category}/marcas/{brand['codigo']}/modelos")["modelos"]
    ranked_models = sorted(
        (
            (
                model_similarity(row["modelo_normalizado"], model["nome"]),
                model,
            )
            for model in models
        ),
        key=lambda candidate: candidate[0],
        reverse=True,
    )
    ranked_models = [
        (similarity, model)
        for similarity, model in ranked_models[:max_model_search]
        if similarity >= 0.28
    ]

    candidates = []
    compatible_models = 0
    for similarity, model in ranked_models:
        years = client.get(
            f"{category}/marcas/{brand['codigo']}/modelos/{model['codigo']}/anos"
        )
        matching_years = available_matching_years(years, fabrication_year)
        if not matching_years:
            continue
        compatible_models += 1
        for year in matching_years:
            candidates.append(
                price_candidate(
                    client,
                    category,
                    brand,
                    model,
                    year,
                    similarity,
                    fabrication_year,
                    row["modelo_normalizado"],
                )
            )
        if compatible_models >= max_model_candidates:
            break

    if not candidates:
        return unmatched_result(row, quantity, "sem_candidato_fipe", "Nenhum candidato com modelo e ano compativeis.")

    candidates.sort(key=lambda candidate: candidate["score"], reverse=True)
    best = candidates[0]
    alternative_models = [
        candidate
        for candidate in candidates
        if candidate["modelo_codigo_api"] != best["modelo_codigo_api"]
    ]
    second_score = alternative_models[0]["score"] if alternative_models else 0
    margin = best["score"] - second_score
    plausible = [
        candidate
        for candidate in candidates
        if candidate is best
        or (
            candidate["score"] >= max(45, best["score"] - 10)
            and candidate["token_overlap"] >= max(0.75, best["token_overlap"] - 0.15)
        )
    ]
    prices = [
        candidate["valor_numero"]
        for candidate in plausible
        if candidate["valor_numero"] is not None
    ]
    match_confidence = confidence(best["score"], margin)
    return {
        "assinatura_veiculo": signature(row),
        "categoria_fipe": category,
        "marca_modelo_original": row["marca_modelo_original"],
        "ano_fabricacao_veiculo": row["ano_fabricacao_veiculo"],
        "marca_normalizada": row["marca_normalizada"],
        "modelo_normalizado": row["modelo_normalizado"],
        "quantidade_veiculos": quantity,
        "status_match": "enriquecido",
        "confianca_match": match_confidence,
        "score_match": best["score"],
        "margem_segundo_candidato": margin,
        "quantidade_candidatos_plausiveis": len(plausible),
        "marca_fipe": best["marca_fipe"],
        "modelo_fipe_sugerido": best["modelo_fipe"],
        "ano_modelo_fipe_sugerido": best["ano_modelo"],
        "combustivel_fipe_sugerido": best["combustivel"],
        "codigo_fipe_sugerido": best["codigo_fipe"],
        "valor_fipe_sugerido": best["valor"],
        "valor_fipe_min": min(prices) if prices else "",
        "valor_fipe_max": max(prices) if prices else "",
        "valor_fipe_mediano": int(statistics.median(prices)) if prices else "",
        "mes_referencia_fipe": best["mes_referencia"],
        "metodo_match": "marca_modelo_ano_aproximado",
        "observacao": "Faixa calculada com candidatos a ate 10 pontos do melhor match.",
    }


def enrich(
    vehicles_path: Path,
    matches_path: Path,
    output_path: Path,
    api_base_url: str,
    cache_dir: Path,
    limit_signatures: int,
    max_model_candidates: int,
    max_model_search: int,
    category: str | None,
) -> None:
    with vehicles_path.open("r", encoding="utf-8", newline="") as handle:
        vehicles = list(csv.DictReader(handle))

    signature_counts = Counter(signature(row) for row in vehicles)
    rows_by_signature: dict[str, dict[str, str]] = {}
    for row in vehicles:
        rows_by_signature.setdefault(signature(row), row)

    eligible_signatures = (
        vehicle_signature
        for vehicle_signature, _ in signature_counts.most_common()
        if is_eligible_signature(rows_by_signature[vehicle_signature])
        and (
            category is None
            or rows_by_signature[vehicle_signature]["categoria_fipe"] == category
        )
    )
    selected_signatures = list(eligible_signatures)[:limit_signatures]
    client = FipeClient(api_base_url, cache_dir)
    matches = []
    for position, vehicle_signature in enumerate(selected_signatures, start=1):
        row = rows_by_signature[vehicle_signature]
        quantity = signature_counts[vehicle_signature]
        print(
            f"[{position}/{len(selected_signatures)}] {row['marca_modelo_original']} "
            f"({row['ano_fabricacao_veiculo']})"
        )
        matches.append(
            match_signature(
                client, row, quantity, max_model_candidates, max_model_search
            )
        )

    write_csv(matches_path, matches, MATCH_FIELDS)
    matches_by_signature = {row["assinatura_veiculo"]: row for row in matches}
    enriched_rows = []
    enrichment_fields = EXTRACT_FIELDS + [
        field for field in MATCH_FIELDS if field not in EXTRACT_FIELDS and field != "assinatura_veiculo"
    ]
    for vehicle in vehicles:
        match = matches_by_signature.get(signature(vehicle))
        if match:
            enriched_rows.append({**vehicle, **match})
        else:
            enriched_rows.append(
                {
                    **vehicle,
                    "status_match": "nao_processado_amostra",
                    "confianca_match": "sem_match",
                }
            )
    write_csv(output_path, enriched_rows, enrichment_fields)
    print(f"Matches gravados: {len(matches)} em {matches_path}")
    print(f"Veiculos enriquecidos gravados: {len(enriched_rows)} em {output_path}")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    subparsers = parser.add_subparsers(dest="command", required=True)

    extract_parser = subparsers.add_parser("extract", help="Extrai uma linha por id_veiculo.")
    extract_parser.add_argument("--input", type=Path, required=True)
    extract_parser.add_argument("--output", type=Path, required=True)

    enrich_parser = subparsers.add_parser("enrich", help="Enriquece uma amostra com dados FIPE.")
    enrich_parser.add_argument("--vehicles", type=Path, required=True)
    enrich_parser.add_argument("--matches", type=Path, required=True)
    enrich_parser.add_argument("--output", type=Path, required=True)
    enrich_parser.add_argument("--api-base-url", default=DEFAULT_API_BASE_URL)
    enrich_parser.add_argument("--cache-dir", type=Path, default=Path(".cache/fipe"))
    enrich_parser.add_argument("--limit-signatures", type=int, default=20)
    enrich_parser.add_argument("--max-model-candidates", type=int, default=4)
    enrich_parser.add_argument("--max-model-search", type=int, default=50)
    enrich_parser.add_argument(
        "--category", choices=("carros", "motos", "caminhoes"), default=None
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()
    if args.command == "extract":
        extract_unique_vehicles(args.input, args.output)
    elif args.command == "enrich":
        enrich(
            vehicles_path=args.vehicles,
            matches_path=args.matches,
            output_path=args.output,
            api_base_url=args.api_base_url,
            cache_dir=args.cache_dir,
            limit_signatures=args.limit_signatures,
            max_model_candidates=args.max_model_candidates,
            max_model_search=args.max_model_search,
            category=args.category,
        )


''', 'enrich_fipe.py', 'exec'), namespace_enrich_fipe)
modulos['enrich_fipe'] = namespace_enrich_fipe
print('Código incorporado: enrich_fipe.py')


### enrich_family_model.py

Cria a família normalizada do veículo e os mapeamentos auditáveis por marca/modelo.

In [ ]:
modulo_enrich_family_model = types.ModuleType('notebook_enrich_family_model')
sys.modules['notebook_enrich_family_model'] = modulo_enrich_family_model
namespace_enrich_family_model = modulo_enrich_family_model.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Iterable

from enrich_fipe import fipe_category, normalize_brand, normalize_text, split_brand_model


INVALID_ORIGINAL_VALUES = {"", "NA", "NA/NA", "Nao Informado/Nao Informado"}
TRUSTED_CONFIDENCES = {"alta", "media"}
CAR_LIKE_TYPES = {"AUTOMOVEL", "CAMINHONETE", "CAMIONETA", "UTILITARIO"}
ALPHANUMERIC_CAR_FAMILY_PREFIXES = {
    "A",
    "C",
    "D",
    "F",
    "HB",
    "I",
    "IX",
    "J",
    "K",
    "L",
    "Q",
    "QQ",
    "RAV",
    "RS",
    "S",
    "T",
    "UK",
    "V",
    "X",
    "XC",
}

BRAND_ALIASES_EXTRA = {
    "CAOACHERY": "CAOA CHERY",
    "CHERY": "CAOA CHERY",
}

MODEL_PREFIX_ALIASES = {
    "carros": {
        "C CROSS": "COROLLA CROSS",
        "CCROSS": "COROLLA CROSS",
        "COROLLA CROSS": "COROLLA CROSS",
        "CR V": "CR-V",
        "DEL REY": "DEL REY",
        "GRAND SIENA": "GRAND SIENA",
        "GRAND VITARA": "GRAND VITARA",
        "HB 20 S": "HB20S",
        "HILUX SW": "SW4",
        "HILUXSW": "SW4",
        "HR V": "HR-V",
        "NOVA SAVEIRO": "SAVEIRO",
        "NOVO GOL": "GOL",
        "RAV 4": "RAV4",
        "SANTA FE": "SANTA FE",
        "SONG PLUS": "SONG PLUS",
        "T CROSS": "T-CROSS",
        "V DRIVE": "V-DRIVE",
        "WR V": "WR-V",
    },
}

FAMILY_FIELDS = [
    "marca_normalizada",
    "modelo_normalizado",
    "familia_modelo",
    "confianca_familia",
    "metodo_familia",
    "observacao_familia",
]

VEHICLE_FIELDS = [
    "id_veiculo",
    "tipo_veiculo",
    "categoria_fipe",
    "marca_modelo_original",
    *FAMILY_FIELDS,
    "ano_fabricacao_veiculo",
    "quantidade_registros_envolvidos",
    "quantidade_acidentes",
]

MAPPING_FIELDS = [
    "tipo_veiculo",
    "categoria_fipe",
    "marca_modelo_original",
    *FAMILY_FIELDS,
    "quantidade_veiculos",
    "quantidade_registros_detalhados",
    "quantidade_acidentes",
]

RANKING_FIELDS = [
    "familia_modelo",
    "acidentes_distintos",
    "acidentes_fatais_distintos",
    "veiculos_envolvidos",
    "veiculos_envolvidos_acidentes_fatais",
    "proporcao_acidentes_fatais",
]


def write_csv(
    path: Path,
    rows: Iterable[dict[str, Any]],
    fieldnames: list[str],
    delimiter: str = ",",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=fieldnames, delimiter=delimiter, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


def detect_delimiter(path: Path) -> str:
    for encoding in ("utf-8-sig", "latin1"):
        try:
            with path.open("r", encoding=encoding, newline="") as handle:
                first_line = handle.readline()
            return ";" if first_line.count(";") >= first_line.count(",") else ","
        except UnicodeDecodeError:
            continue
    return ";"


def read_csv_rows(path: Path, delimiter: str | None = None) -> tuple[list[str], list[dict[str, str]]]:
    delimiter = delimiter or detect_delimiter(path)
    for encoding in ("utf-8-sig", "latin1"):
        try:
            with path.open("r", encoding=encoding, newline="") as handle:
                reader = csv.DictReader(handle, delimiter=delimiter)
                return reader.fieldnames or [], list(reader)
        except UnicodeDecodeError:
            continue
    with path.open("r", encoding="latin1", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        return reader.fieldnames or [], list(reader)


def accident_key(row: dict[str, str]) -> str:
    accident_id = row.get("id", "")
    year = row.get("ano_arquivo", "")
    return f"{year}:{accident_id}" if year else accident_id


def vehicle_key(row: dict[str, str]) -> str:
    vehicle_id = row.get("id_veiculo", "")
    if vehicle_id in {"", "NA"}:
        return vehicle_id
    year = row.get("ano_arquivo", "")
    return f"{year}:{vehicle_id}" if year else vehicle_id


def sort_vehicle_id(value: str) -> tuple[str, int, str]:
    suffix = value.rsplit(":", maxsplit=1)[-1]
    prefix = value[: -len(suffix)]
    return prefix, int(suffix) if suffix.isdigit() else -1, suffix


def normalized_brand(value: str) -> str:
    brand = normalize_brand(value)
    return BRAND_ALIASES_EXTRA.get(brand, brand)


def mapping_key(row: dict[str, str]) -> tuple[str, str]:
    return row.get("tipo_veiculo", ""), row.get("marca", "")


def is_invalid_original(value: str) -> bool:
    return normalize_text(value) in {
        normalize_text(invalid_value) for invalid_value in INVALID_ORIGINAL_VALUES
    }


def first_family_token(model: str) -> str:
    tokens = normalize_text(model).split()
    if not tokens:
        return ""
    if tokens[0].isdigit() and len(tokens) >= 2 and tokens[1].isdigit():
        return f"{tokens[0]}.{tokens[1]}"
    return tokens[0]


def resolve_family(category: str, model: str) -> tuple[str, str, str]:
    normalized_model = normalize_text(model)
    for prefix, family in sorted(
        MODEL_PREFIX_ALIASES.get(category, {}).items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        if normalized_model.startswith(prefix):
            return family, "alta", f"prefixo_curado:{prefix}"

    tokens = normalized_model.split()
    family = first_family_token(normalized_model)
    if not family:
        return "", "sem_match", "modelo_ausente"
    if family[0].isdigit():
        if category == "carros":
            return family, "media", "prefixo_numerico_automovel"
        return family, "baixa", "prefixo_numerico_modelo"
    if (
        category == "carros"
        and len(tokens) >= 2
        and tokens[0] in ALPHANUMERIC_CAR_FAMILY_PREFIXES
        and tokens[1].isdigit()
    ):
        return f"{tokens[0]}{tokens[1]}", "media", "prefixo_alfanumerico_automovel"
    return family, "media", "primeiro_token_modelo"


def family_fields(vehicle_type: str, original: str) -> dict[str, str]:
    if is_invalid_original(original):
        return {
            "marca_normalizada": "",
            "modelo_normalizado": "",
            "familia_modelo": "",
            "confianca_familia": "sem_match",
            "metodo_familia": "dados_insuficientes",
            "observacao_familia": "Marca/modelo ausente ou nao informado.",
        }

    brand, model = split_brand_model(original)
    brand = normalized_brand(brand)
    model = normalize_text(model)
    category = fipe_category(vehicle_type)
    family, confidence, method = resolve_family(category, model)
    if not brand or not family:
        return {
            "marca_normalizada": brand,
            "modelo_normalizado": model,
            "familia_modelo": "",
            "confianca_familia": "sem_match",
            "metodo_familia": "dados_insuficientes",
            "observacao_familia": "Nao foi possivel extrair marca e familia.",
        }
    return {
        "marca_normalizada": brand,
        "modelo_normalizado": model,
        "familia_modelo": f"{brand}/{family}",
        "confianca_familia": confidence,
        "metodo_familia": method,
        "observacao_familia": (
            "Familia derivada localmente do texto PRF; revisar antes de uso causal."
        ),
    }


def read_fatal_ids(accidents_path: Path) -> set[str]:
    _, rows = read_csv_rows(accidents_path)
    return {
        accident_key(row)
        for row in rows
        if row.get("classificacao_acidente", "").strip() == "Com Vítimas Fatais"
    }


def build_enrichment(
    input_path: Path,
    accidents_path: Path,
    vehicles_output: Path,
    mapping_output: Path,
    detailed_output: Path,
    ranking_output: Path,
) -> None:
    input_fields, detailed_rows = read_csv_rows(input_path)

    fatal_ids = read_fatal_ids(accidents_path)
    mappings: dict[tuple[str, str], dict[str, str]] = {}
    mapping_accidents: dict[tuple[str, str], set[str]] = defaultdict(set)
    mapping_vehicles: dict[tuple[str, str], set[str]] = defaultdict(set)
    mapping_detail_counts: Counter[tuple[str, str]] = Counter()
    vehicles: dict[str, dict[str, Any]] = {}

    for row in detailed_rows:
        key = mapping_key(row)
        mappings.setdefault(key, family_fields(row.get("tipo_veiculo", ""), row.get("marca", "")))
        mapping_accidents[key].add(accident_key(row))
        mapping_detail_counts[key] += 1
        vehicle_id = vehicle_key(row)
        if vehicle_id not in {"", "NA"}:
            mapping_vehicles[key].add(vehicle_id)
            vehicle = vehicles.setdefault(
                vehicle_id,
                {
                    "id_veiculo": vehicle_id,
                    "tipo_veiculo": row.get("tipo_veiculo", ""),
                    "categoria_fipe": fipe_category(row.get("tipo_veiculo", "")),
                    "marca_modelo_original": row.get("marca", ""),
                    **mappings[key],
                    "ano_fabricacao_veiculo": row.get("ano_fabricacao_veiculo", ""),
                    "accident_ids": set(),
                    "quantidade_registros_envolvidos": 0,
                },
            )
            vehicle["accident_ids"].add(accident_key(row))
            vehicle["quantidade_registros_envolvidos"] += 1

    vehicle_rows = [
        {
            **vehicle,
            "quantidade_acidentes": len(vehicle["accident_ids"]),
        }
        for vehicle in vehicles.values()
    ]
    vehicle_rows.sort(key=lambda row: sort_vehicle_id(row["id_veiculo"]))
    write_csv(vehicles_output, vehicle_rows, VEHICLE_FIELDS)

    mapping_rows = [
        {
            "tipo_veiculo": key[0],
            "categoria_fipe": fipe_category(key[0]),
            "marca_modelo_original": key[1],
            **family,
            "quantidade_veiculos": len(mapping_vehicles[key]),
            "quantidade_registros_detalhados": mapping_detail_counts[key],
            "quantidade_acidentes": len(mapping_accidents[key]),
        }
        for key, family in mappings.items()
    ]
    mapping_rows.sort(
        key=lambda row: (-row["quantidade_veiculos"], row["marca_modelo_original"])
    )
    write_csv(mapping_output, mapping_rows, MAPPING_FIELDS)

    enriched_detailed_rows = [
        {**row, **mappings[mapping_key(row)]}
        for row in detailed_rows
    ]
    write_csv(detailed_output, enriched_detailed_rows, input_fields + FAMILY_FIELDS, delimiter=";")

    ranking = build_fatal_ranking(enriched_detailed_rows, fatal_ids)
    write_csv(ranking_output, ranking, RANKING_FIELDS)

    print(f"Veiculos unicos enriquecidos: {len(vehicle_rows)} em {vehicles_output}")
    print(f"Mapeamentos auditaveis: {len(mapping_rows)} em {mapping_output}")
    print(f"Registros detalhados enriquecidos: {len(enriched_detailed_rows)} em {detailed_output}")
    print(f"Familias no ranking: {len(ranking)} em {ranking_output}")


def build_fatal_ranking(
    rows: list[dict[str, str]], fatal_ids: set[str]
) -> list[dict[str, Any]]:
    accident_ids: dict[str, set[str]] = defaultdict(set)
    fatal_accident_ids: dict[str, set[str]] = defaultdict(set)
    vehicle_ids: dict[str, set[tuple[str, str]]] = defaultdict(set)
    fatal_vehicle_ids: dict[str, set[tuple[str, str]]] = defaultdict(set)

    for row in rows:
        if (
            normalize_text(row.get("tipo_veiculo", "")) not in CAR_LIKE_TYPES
            or row.get("confianca_familia", "") not in TRUSTED_CONFIDENCES
            or not row.get("familia_modelo", "")
        ):
            continue
        family = row["familia_modelo"]
        accident_id = accident_key(row)
        accident_ids[family].add(accident_id)
        if accident_id in fatal_ids:
            fatal_accident_ids[family].add(accident_id)
        vehicle_id = vehicle_key(row)
        if vehicle_id not in {"", "NA"}:
            accident_vehicle_key = accident_id, vehicle_id
            vehicle_ids[family].add(accident_vehicle_key)
            if accident_id in fatal_ids:
                fatal_vehicle_ids[family].add(accident_vehicle_key)

    ranking = []
    for family, family_accidents in accident_ids.items():
        fatal_count = len(fatal_accident_ids[family])
        ranking.append(
            {
                "familia_modelo": family,
                "acidentes_distintos": len(family_accidents),
                "acidentes_fatais_distintos": fatal_count,
                "veiculos_envolvidos": len(vehicle_ids[family]),
                "veiculos_envolvidos_acidentes_fatais": len(fatal_vehicle_ids[family]),
                "proporcao_acidentes_fatais": (
                    f"{fatal_count / len(family_accidents):.6f}"
                    if family_accidents
                    else ""
                ),
            }
        )
    ranking.sort(
        key=lambda row: (
            -row["acidentes_fatais_distintos"],
            -row["acidentes_distintos"],
            row["familia_modelo"],
        )
    )
    return ranking


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--accidents", type=Path, required=True)
    parser.add_argument("--vehicles-output", type=Path, required=True)
    parser.add_argument("--mapping-output", type=Path, required=True)
    parser.add_argument("--detailed-output", type=Path, required=True)
    parser.add_argument("--ranking-output", type=Path, required=True)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    build_enrichment(
        input_path=args.input,
        accidents_path=args.accidents,
        vehicles_output=args.vehicles_output,
        mapping_output=args.mapping_output,
        detailed_output=args.detailed_output,
        ranking_output=args.ranking_output,
    )


''', 'enrich_family_model.py', 'exec'), namespace_enrich_family_model)
modulos['enrich_family_model'] = namespace_enrich_family_model
print('Código incorporado: enrich_family_model.py')


### enrich_latin_ncap.py

Consulta/cacheia resultados do Latin NCAP e associa os testes às famílias e anos dos veículos.

In [ ]:
modulo_enrich_latin_ncap = types.ModuleType('notebook_enrich_latin_ncap')
sys.modules['notebook_enrich_latin_ncap'] = modulo_enrich_latin_ncap
namespace_enrich_latin_ncap = modulo_enrich_latin_ncap.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
import json
import re
import unicodedata
import urllib.parse
import urllib.request
from collections import defaultdict
from datetime import date
from pathlib import Path
from typing import Any, Iterable

from enrich_family_model import family_fields
from enrich_fipe import normalize_text, parse_year


LATIN_NCAP_BASE_URL = "https://www.latinncap.com"
LATIN_NCAP_RESULTS_URL = f"{LATIN_NCAP_BASE_URL}/get_res.php"
PROTOCOLS = ("a20", "a19", "a15")
MAX_YEAR_DISTANCE = 2

LATIN_NCAP_FIELDS = [
    "latin_ncap_id",
    "latin_ncap_protocolo",
    "latin_ncap_data_teste",
    "latin_ncap_ano_teste",
    "latin_ncap_marca",
    "latin_ncap_nome_original",
    "latin_ncap_variante",
    "familias_modelo_latin_ncap",
    "latin_ncap_airbags",
    "latin_ncap_estrelas",
    "latin_ncap_estrelas_adulto",
    "latin_ncap_estrelas_crianca",
    "latin_ncap_score_adulto",
    "latin_ncap_score_adulto_max",
    "latin_ncap_score_crianca",
    "latin_ncap_score_crianca_max",
    "latin_ncap_percentual_adulto",
    "latin_ncap_percentual_crianca",
    "latin_ncap_percentual_pedestres",
    "latin_ncap_percentual_assistentes",
    "latin_ncap_url",
]

ENRICHMENT_FIELDS = [
    "status_latin_ncap",
    "confianca_latin_ncap",
    "latin_ncap_protocolo",
    "latin_ncap_ano_teste",
    "latin_ncap_distancia_ano",
    "latin_ncap_quantidade_testes_compativeis",
    "latin_ncap_airbags_min",
    "latin_ncap_airbags_max",
    "latin_ncap_estrelas_min",
    "latin_ncap_estrelas_max",
    "latin_ncap_estrelas_adulto_min",
    "latin_ncap_estrelas_adulto_max",
    "latin_ncap_estrelas_crianca_min",
    "latin_ncap_estrelas_crianca_max",
    "latin_ncap_score_adulto_min",
    "latin_ncap_score_adulto_max",
    "latin_ncap_score_crianca_min",
    "latin_ncap_score_crianca_max",
    "latin_ncap_percentual_adulto_min",
    "latin_ncap_percentual_adulto_max",
    "latin_ncap_percentual_crianca_min",
    "latin_ncap_percentual_crianca_max",
    "latin_ncap_urls",
    "observacao_latin_ncap",
]

MAPPING_FIELDS = [
    "familia_modelo",
    "ano_fabricacao_veiculo",
    "quantidade_veiculos",
    *ENRICHMENT_FIELDS,
]

RESULT_FAMILY_OVERRIDES = {
    "CHEVROLET Corsa Classic": ["CHEVROLET/CORSA", "CHEVROLET/CLASSIC"],
    "CHEVROLET Onix/Prisma": ["CHEVROLET/ONIX", "CHEVROLET/PRISMA"],
    "FIAT Argo / Cronos": ["FIAT/ARGO", "FIAT/CRONOS"],
    "FORD Ka / Figo": ["FORD/KA", "FORD/FIGO"],
    "KIA Picanto / Morning": ["KIA/PICANTO", "KIA/MORNING"],
    "MITSUBISHI L200 / Triton": ["MITSUBISHI/L200"],
    "NISSAN Frontier / NP300 Navara": ["NISSAN/FRONTIER"],
    "RENAULT Sandero / Logan": ["RENAULT/SANDERO", "RENAULT/LOGAN"],
    "RENAULT Sandero / Logan / Stepway": [
        "RENAULT/SANDERO",
        "RENAULT/LOGAN",
        "RENAULT/STEPWAY",
    ],
    "TOYOTA Hilux / SW4 / Fortuner": ["TOYOTA/HILUX", "TOYOTA/SW4", "TOYOTA/FORTUNER"],
    "TOYOTA Hilux Double Cab / SW4": ["TOYOTA/HILUX", "TOYOTA/SW4"],
    "VOLKSWAGEN Jetta / Vento": ["VOLKSWAGEN/JETTA", "VOLKSWAGEN/VENTO"],
    "VOLKSWAGEN New Polo / Polo Track": ["VOLKSWAGEN/POLO"],
    "VOLKSWAGEN Suran/Fox": ["VOLKSWAGEN/SURAN", "VOLKSWAGEN/FOX"],
    "VOLKSWAGEN Vento / Polo Sedan": ["VOLKSWAGEN/VENTO", "VOLKSWAGEN/POLO"],
}


def write_csv(
    path: Path,
    rows: Iterable[dict[str, Any]],
    fieldnames: list[str],
    delimiter: str = ",",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=fieldnames, delimiter=delimiter, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


def vehicle_key(row: dict[str, str]) -> str:
    vehicle_id = row.get("id_veiculo", "")
    if vehicle_id in {"", "NA"}:
        return vehicle_id
    year = row.get("ano_arquivo", "")
    return f"{year}:{vehicle_id}" if year else vehicle_id


def fetch_results(cache_path: Path, refresh: bool = False) -> dict[str, Any]:
    if cache_path.exists() and not refresh:
        return json.loads(cache_path.read_text(encoding="utf-8"))

    body = urllib.parse.urlencode(
        {
            "lg": "en",
            "protocolo": ",".join(PROTOCOLS),
        }
    ).encode("utf-8")
    request = urllib.request.Request(
        LATIN_NCAP_RESULTS_URL,
        data=body,
        headers={"User-Agent": "prf-analise-latinncap-enrichment/1.0"},
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        results = json.load(response)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(
        json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return results


def parse_airbags(value: str) -> int | None:
    if re.search(r"\bNO\s+Airbags?\b", value, flags=re.IGNORECASE):
        return 0
    match = re.search(r"[+-]\s*(\d+)\s*Airbags?\b", value, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def clean_result_name(value: str, brand: str) -> str:
    value = value.strip()
    value = re.sub(rf"^{re.escape(brand)}\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(
        r"\s*(?:[+-]\s*\d+\s*Airbags?|\-\s*NO\s+Airbags?).*$",
        "",
        value,
        flags=re.IGNORECASE,
    )
    value = re.sub(r"\s*\([^)]*\)\s*$", "", value)
    value = re.sub(r"\s+from\s+.*$", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\s+VIN\s+.*$", "", value, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", value).strip()


def result_family_key(brand: str, model: str) -> str:
    key = normalize_text(f"{brand} {model}")
    return key.replace(" ", "")


def result_families(brand: str, model: str) -> list[str]:
    normalized_brand = family_fields("Automovel", f"{brand}/X")["marca_normalizada"]
    cleaned_model = re.sub(r"^(NEW|NOVO)\s+", "", model, flags=re.IGNORECASE)
    normalized_override_key = result_family_key(normalized_brand, cleaned_model)
    for override, families in RESULT_FAMILY_OVERRIDES.items():
        if result_family_key(*override.split(" ", 1)) == normalized_override_key:
            return families

    result = family_fields("Automovel", f"{normalized_brand}/{cleaned_model}")
    family = result["familia_modelo"]
    return [family] if family else []


def absolute_url(path: str) -> str:
    return urllib.parse.urljoin(f"{LATIN_NCAP_BASE_URL}/", path)


def result_id_from_url(url: str) -> str:
    match = re.search(r"/result/(\d+)/", url)
    return match.group(1) if match else ""


def flatten_result(
    protocol: str,
    item: dict[str, Any],
    variant: dict[str, Any] | None = None,
) -> dict[str, Any]:
    name = item.get("nombre", "")
    url = item.get("link", "")
    variant_name = ""
    if variant:
        variant_name = variant.get("titulo_marca", "")
        url = variant.get("link_vermas", "")
    model = clean_result_name(name, item.get("tit_marca", ""))
    if variant:
        model = re.sub(r"\s*\([^)]*\)\s*$", "", model).strip()
    families = result_families(item.get("tit_marca", ""), model)
    airbags = parse_airbags(variant_name or name)
    return {
        "latin_ncap_id": result_id_from_url(url) or item.get("id", ""),
        "latin_ncap_protocolo": protocol,
        "latin_ncap_data_teste": item.get("fecha", ""),
        "latin_ncap_ano_teste": (item.get("fecha", "") or "")[:4],
        "latin_ncap_marca": item.get("tit_marca", ""),
        "latin_ncap_nome_original": name,
        "latin_ncap_variante": variant_name,
        "familias_modelo_latin_ncap": "|".join(families),
        "latin_ncap_airbags": airbags if airbags is not None else "",
        "latin_ncap_estrelas": (
            variant.get("a20_main_stars", "") if variant else item.get("a20_main_stars", "")
        ),
        "latin_ncap_estrelas_adulto": item.get("star_adult", ""),
        "latin_ncap_estrelas_crianca": item.get("star_child", ""),
        "latin_ncap_score_adulto": item.get("score_adult", ""),
        "latin_ncap_score_adulto_max": item.get("protocolo_a_max", ""),
        "latin_ncap_score_crianca": item.get("score_child", ""),
        "latin_ncap_score_crianca_max": item.get("protocolo_c_max", ""),
        "latin_ncap_percentual_adulto": (
            variant.get("a20_percentage_a", "") if variant else item.get("a20_percentage_a", "")
        ),
        "latin_ncap_percentual_crianca": (
            variant.get("a20_percentage_c", "") if variant else item.get("a20_percentage_c", "")
        ),
        "latin_ncap_percentual_pedestres": (
            variant.get("a20_percentage_p", "") if variant else item.get("a20_percentage_p", "")
        ),
        "latin_ncap_percentual_assistentes": (
            variant.get("a20_percentage_s", "") if variant else item.get("a20_percentage_s", "")
        ),
        "latin_ncap_url": absolute_url(url),
    }


def flatten_results(payload: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    for protocol in PROTOCOLS:
        for item in payload.get(protocol, {}).get("data", []):
            pair = item.get("pareja") or {}
            if pair:
                rows.append(flatten_result(protocol, item, pair.get("a", {})))
                rows.append(flatten_result(protocol, item, pair.get("b", {})))
            else:
                rows.append(flatten_result(protocol, item))
    rows.sort(key=lambda row: (row["latin_ncap_id"], row["latin_ncap_url"]))
    return rows


def numeric_values(rows: list[dict[str, Any]], field: str) -> list[float]:
    values = []
    for row in rows:
        value = row.get(field, "")
        if value in {"", None}:
            continue
        try:
            values.append(float(value))
        except (TypeError, ValueError):
            continue
    return values


def range_fields(
    rows: list[dict[str, Any]], source_field: str, output_prefix: str
) -> dict[str, Any]:
    values = numeric_values(rows, source_field)
    if not values:
        return {f"{output_prefix}_min": "", f"{output_prefix}_max": ""}
    normalized = [int(value) if value.is_integer() else value for value in values]
    return {
        f"{output_prefix}_min": min(normalized),
        f"{output_prefix}_max": max(normalized),
    }


def unmatched(status: str, observation: str) -> dict[str, Any]:
    return {
        "status_latin_ncap": status,
        "confianca_latin_ncap": "sem_match",
        "observacao_latin_ncap": observation,
    }


def choose_results(
    family: str,
    fabrication_year: str,
    results_by_family: dict[str, list[dict[str, Any]]],
) -> dict[str, Any]:
    year = parse_year(fabrication_year)
    if not family:
        return unmatched("familia_modelo_ausente", "Veiculo sem familia_modelo utilizavel.")
    candidates = results_by_family.get(family, [])
    if not candidates:
        return unmatched("sem_teste_latin_ncap", "Familia sem teste Latin NCAP publicado.")
    if not year:
        return unmatched("ano_fabricacao_invalido", "Ano de fabricacao ausente ou invalido.")

    candidates_with_distance = [
        (abs(year - int(candidate["latin_ncap_ano_teste"])), candidate)
        for candidate in candidates
        if candidate.get("latin_ncap_ano_teste", "").isdigit()
    ]
    if not candidates_with_distance:
        return unmatched("teste_sem_ano", "Teste Latin NCAP sem ano publicado.")
    minimum_distance = min(distance for distance, _ in candidates_with_distance)
    if minimum_distance > MAX_YEAR_DISTANCE:
        return unmatched(
            "familia_com_teste_sem_ano_compativel",
            f"Teste mais proximo esta a {minimum_distance} anos do ano de fabricacao.",
        )

    selected = [
        candidate
        for distance, candidate in candidates_with_distance
        if distance == minimum_distance
    ]
    protocols = sorted({candidate["latin_ncap_protocolo"] for candidate in selected})
    confidence = "media" if minimum_distance == 0 and len(selected) == 1 else "baixa"
    observation = (
        "Associacao aproximada por familia e ano; nao comprova versao, VIN ou configuracao."
    )
    if len(selected) > 1:
        observation += " Existem configuracoes Latin NCAP concorrentes; use as faixas."

    result = {
        "status_latin_ncap": "enriquecido",
        "confianca_latin_ncap": confidence,
        "latin_ncap_protocolo": "|".join(protocols),
        "latin_ncap_ano_teste": "|".join(
            sorted({candidate["latin_ncap_ano_teste"] for candidate in selected})
        ),
        "latin_ncap_distancia_ano": minimum_distance,
        "latin_ncap_quantidade_testes_compativeis": len(selected),
        "latin_ncap_urls": "|".join(
            sorted({candidate["latin_ncap_url"] for candidate in selected})
        ),
        "observacao_latin_ncap": observation,
    }
    result.update(range_fields(selected, "latin_ncap_airbags", "latin_ncap_airbags"))
    result.update(range_fields(selected, "latin_ncap_estrelas", "latin_ncap_estrelas"))
    result.update(
        range_fields(selected, "latin_ncap_estrelas_adulto", "latin_ncap_estrelas_adulto")
    )
    result.update(
        range_fields(selected, "latin_ncap_estrelas_crianca", "latin_ncap_estrelas_crianca")
    )
    result.update(range_fields(selected, "latin_ncap_score_adulto", "latin_ncap_score_adulto"))
    result.update(range_fields(selected, "latin_ncap_score_crianca", "latin_ncap_score_crianca"))
    result.update(
        range_fields(selected, "latin_ncap_percentual_adulto", "latin_ncap_percentual_adulto")
    )
    result.update(
        range_fields(selected, "latin_ncap_percentual_crianca", "latin_ncap_percentual_crianca")
    )
    return result


def enrich_dataset(
    input_path: Path,
    cache_path: Path,
    source_output: Path,
    mapping_output: Path,
    output_path: Path,
    refresh: bool,
) -> None:
    payload = fetch_results(cache_path, refresh=refresh)
    latin_results = flatten_results(payload)
    write_csv(source_output, latin_results, LATIN_NCAP_FIELDS)

    results_by_family: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for latin_result in latin_results:
        for family in latin_result["familias_modelo_latin_ncap"].split("|"):
            if family:
                results_by_family[family].append(latin_result)

    with input_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=";")
        input_fields = reader.fieldnames or []
        detailed_rows = list(reader)

    signature_counts: dict[tuple[str, str], set[str]] = defaultdict(set)
    for row in detailed_rows:
        signature_counts[
            (row.get("familia_modelo", ""), row.get("ano_fabricacao_veiculo", ""))
        ].add(vehicle_key(row))

    mappings = []
    mappings_by_signature = {}
    for signature, vehicle_ids in signature_counts.items():
        family, fabrication_year = signature
        enrichment = choose_results(family, fabrication_year, results_by_family)
        mapping = {
            "familia_modelo": family,
            "ano_fabricacao_veiculo": fabrication_year,
            "quantidade_veiculos": len(vehicle_ids - {"", "NA"}),
            **enrichment,
        }
        mappings.append(mapping)
        mappings_by_signature[signature] = enrichment
    mappings.sort(
        key=lambda row: (
            -row["quantidade_veiculos"],
            row["familia_modelo"],
            row["ano_fabricacao_veiculo"],
        )
    )
    write_csv(mapping_output, mappings, MAPPING_FIELDS)

    enriched_rows = [
        {
            **row,
            **mappings_by_signature[
                (row.get("familia_modelo", ""), row.get("ano_fabricacao_veiculo", ""))
            ],
        }
        for row in detailed_rows
    ]
    write_csv(output_path, enriched_rows, input_fields + ENRICHMENT_FIELDS, delimiter=";")

    print(f"Resultados Latin NCAP coletados: {len(latin_results)} em {source_output}")
    print(f"Assinaturas familia+ano processadas: {len(mappings)} em {mapping_output}")
    print(f"Registros detalhados enriquecidos: {len(enriched_rows)} em {output_path}")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--cache", type=Path, default=Path(".cache/latinncap/results.json"))
    parser.add_argument("--source-output", type=Path, required=True)
    parser.add_argument("--mapping-output", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--refresh", action="store_true")
    return parser


def main() -> None:
    args = build_parser().parse_args()
    enrich_dataset(
        input_path=args.input,
        cache_path=args.cache,
        source_output=args.source_output,
        mapping_output=args.mapping_output,
        output_path=args.output,
        refresh=args.refresh,
    )


''', 'enrich_latin_ncap.py', 'exec'), namespace_enrich_latin_ncap)
modulos['enrich_latin_ncap'] = namespace_enrich_latin_ncap
print('Código incorporado: enrich_latin_ncap.py')


### enrich_fipe_local.py

Associa os veículos à fotografia local da FIPE e gera os arquivos de mapeamento e resultado.

In [ ]:
modulo_enrich_fipe_local = types.ModuleType('notebook_enrich_fipe_local')
sys.modules['notebook_enrich_fipe_local'] = modulo_enrich_fipe_local
namespace_enrich_fipe_local = modulo_enrich_fipe_local.__dict__
exec(compile(r'''
#!/usr/bin/env python3
             
from __future__ import annotations

import argparse
import csv
import statistics
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Iterable

from enrich_fipe import (
    INVALID_VALUES,
    confidence,
    fipe_category,
    normalize_brand,
    normalize_text,
    parse_year,
    split_brand_model,
)


FIPEX_SOURCE_URL = (
    "https://huggingface.co/datasets/alanwgt/fipex-veiculos-brasil/"
    "resolve/main/2026/05/fipex-prices.csv"
)

SOURCE_CATEGORY = {
    "CARRO": "carros",
    "MOTO": "motos",
    "CAMINHAO": "caminhoes",
}

FIPEX_BRAND_ALIASES = {
    "GM CHEVROLET": "CHEVROLET",
    "KIA MOTORS": "KIA",
    "VW VOLKSWAGEN": "VOLKSWAGEN",
}

ENRICHMENT_FIELDS = [
    "status_fipe",
    "confianca_fipe",
    "fipe_categoria",
    "fipe_score_match",
    "fipe_margem_segundo_candidato",
    "fipe_quantidade_candidatos_plausiveis",
    "fipe_marca_sugerida",
    "fipe_modelo_sugerido",
    "fipe_ano_modelo_sugerido",
    "fipe_combustivel_sugerido",
    "fipe_codigo_sugerido",
    "fipe_valor_sugerido",
    "fipe_valor_min",
    "fipe_valor_max",
    "fipe_valor_mediano",
    "fipe_referencia",
    "fipe_metodo_match",
    "fipe_fonte",
    "observacao_fipe",
]

MAPPING_FIELDS = [
    "assinatura_fipe",
    "tipo_veiculo",
    "marca_modelo_original",
    "ano_fabricacao_veiculo",
    "marca_normalizada_fipe",
    "modelo_normalizado_fipe",
    "quantidade_veiculos",
    *ENRICHMENT_FIELDS,
]

VEHICLE_FIELDS = [
    "id_veiculo",
    "tipo_veiculo",
    "marca_modelo_original",
    "ano_fabricacao_veiculo",
    *ENRICHMENT_FIELDS,
]


def write_csv(
    path: Path,
    rows: Iterable[dict[str, Any]],
    fieldnames: list[str],
    delimiter: str = ",",
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=fieldnames, delimiter=delimiter, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)


def vehicle_key(row: dict[str, str]) -> str:
    vehicle_id = row.get("id_veiculo", "")
    if vehicle_id in {"", "NA"}:
        return vehicle_id
    year = row.get("ano_arquivo", "")
    return f"{year}:{vehicle_id}" if year else vehicle_id


def sort_vehicle_id(value: str) -> tuple[str, int, str]:
    suffix = value.rsplit(":", maxsplit=1)[-1]
    prefix = value[: -len(suffix)]
    return prefix, int(suffix) if suffix.isdigit() else -1, suffix


def normalize_fipex_brand(value: str) -> str:
    normalized = normalize_brand(value)
    return FIPEX_BRAND_ALIASES.get(normalized, normalized)


def source_category(value: str) -> str:
    return SOURCE_CATEGORY.get(normalize_text(value), "")


def source_price_reais(row: dict[str, str]) -> int:
    return round(int(row["valor_centavos"]) / 100)


def token_set(value: str) -> set[str]:
    return set(normalize_text(value).split())


def token_overlap_sets(source_tokens: set[str], target_tokens: set[str]) -> float:
    if not source_tokens:
        return 0.0
    return len(source_tokens & target_tokens) / len(source_tokens)


def model_similarity_fast(
    source_normalized: str,
    target_normalized: str,
    source_tokens: set[str],
    target_tokens: set[str],
) -> float:
    sequence_ratio = SequenceMatcher(None, source_normalized, target_normalized).ratio()
    overlap = token_overlap_sets(source_tokens, target_tokens)
    prefix_bonus = 0.10 if target_normalized.startswith(source_normalized) else 0.0
    return min(1.0, (0.55 * overlap) + (0.45 * sequence_ratio) + prefix_bonus)


def vehicle_signature(row: dict[str, str]) -> str:
    brand, model = split_brand_model(row.get("marca", ""))
    values = (
        fipe_category(row.get("tipo_veiculo", "")),
        normalize_brand(brand),
        normalize_text(model),
        row.get("ano_fabricacao_veiculo", ""),
    )
    return "|".join(values)


def load_snapshot(
    path: Path,
) -> tuple[dict[tuple[Any, ...], list[dict[str, Any]]], str]:
    index: dict[tuple[Any, ...], list[dict[str, Any]]] = defaultdict(list)
    references = set()
    with path.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle, delimiter="\t"):
            category = source_category(row["tipo_veiculo"])
            brand = normalize_fipex_brand(row["nome_marca"])
            year = parse_year(row["ano_modelo"])
            if not category or not brand or not year:
                continue
            references.add(f"{row['ano_referencia']}-{int(row['mes_referencia']):02d}")
            indexed_row = {
                **row,
                "fipe_categoria": category,
                "marca_normalizada_fipex": brand,
                "modelo_normalizado_fipex": normalize_text(row["nome_modelo"]),
                "modelo_tokens_fipex": token_set(row["nome_modelo"]),
                "ano_modelo_numero": year,
                "valor_reais": source_price_reais(row),
            }
            index[(category, brand)].append(indexed_row)
            index[(category, brand, year)].append(indexed_row)
    if len(references) != 1:
        raise ValueError(f"Snapshot deve ter exatamente uma referencia mensal: {references}")
    return index, references.pop()


def unmatched(
    status: str,
    category: str,
    reference: str,
    observation: str,
) -> dict[str, Any]:
    return {
        "status_fipe": status,
        "confianca_fipe": "sem_match",
        "fipe_categoria": category,
        "fipe_referencia": reference,
        "fipe_metodo_match": "marca_modelo_ano_aproximado_snapshot_local",
        "fipe_fonte": FIPEX_SOURCE_URL,
        "observacao_fipe": observation,
    }


def match_vehicle(
    row: dict[str, str],
    source_index: dict[tuple[Any, ...], list[dict[str, Any]]],
    reference: str,
) -> dict[str, Any]:
    category = fipe_category(row.get("tipo_veiculo", ""))
    original = row.get("marca", "")
    fabrication_year = parse_year(row.get("ano_fabricacao_veiculo", ""))
    brand, model = split_brand_model(original)
    normalized_brand = normalize_brand(brand)
    normalized_model = normalize_text(model)
    normalized_model_tokens = set(normalized_model.split())

    if not category:
        return unmatched(
            "categoria_nao_suportada",
            category,
            reference,
            "Tipo de veiculo fora das categorias FIPE usadas.",
        )
    if normalize_text(original) in INVALID_VALUES or not normalized_model:
        return unmatched(
            "dados_insuficientes",
            category,
            reference,
            "Marca/modelo ausente ou nao informado.",
        )
    if not fabrication_year:
        return unmatched(
            "dados_insuficientes",
            category,
            reference,
            "Ano de fabricacao ausente ou invalido.",
        )

    brand_rows = (
        source_index.get((category, normalized_brand, fabrication_year), [])
        + source_index.get((category, normalized_brand, fabrication_year + 1), [])
    )
    if not brand_rows:
        brand_rows = source_index.get((category, normalized_brand), [])
    if not brand_rows:
        return unmatched(
            "marca_sem_match",
            category,
            reference,
            "Marca PRF nao associada a uma marca no snapshot FIPE.",
        )

    candidates = []
    for source_row in brand_rows:
        model_year = source_row["ano_modelo_numero"]
        if model_year not in {fabrication_year, fabrication_year + 1}:
            continue
        target_normalized = source_row.get("modelo_normalizado_fipex") or normalize_text(
            source_row["nome_modelo"]
        )
        target_tokens = source_row.get("modelo_tokens_fipex") or set(
            target_normalized.split()
        )
        if (
            normalized_model_tokens
            and target_tokens
            and not (normalized_model_tokens & target_tokens)
            and normalized_model not in target_normalized
            and target_normalized not in normalized_model
        ):
            continue
        similarity = model_similarity_fast(
            normalized_model,
            target_normalized,
            normalized_model_tokens,
            target_tokens,
        )
        year_points = 15 if model_year == fabrication_year else 10
        candidates.append(
            {
                **source_row,
                "similarity": similarity,
                "token_overlap": token_overlap_sets(normalized_model_tokens, target_tokens),
                "score": 30 + round(similarity * 50) + year_points,
            }
        )

    if not candidates:
        return unmatched(
            "sem_candidato_ano",
            category,
            reference,
            "Nenhum candidato FIPE com ano-modelo igual a fabricacao ou fabricacao + 1.",
        )

    candidates.sort(key=lambda candidate: candidate["score"], reverse=True)
    best = candidates[0]
    if best["score"] < 45:
        return unmatched(
            "sem_candidato_modelo",
            category,
            reference,
            "Nenhum modelo FIPE suficientemente compativel com o texto PRF.",
        )

    alternative_models = [
        candidate
        for candidate in candidates
        if candidate["codigo_fipe"] != best["codigo_fipe"]
    ]
    second_score = alternative_models[0]["score"] if alternative_models else 0
    margin = best["score"] - second_score
    plausible = [
        candidate
        for candidate in candidates
        if candidate is best
        or (
            candidate["score"] >= max(45, best["score"] - 10)
            and candidate["token_overlap"] >= max(0.75, best["token_overlap"] - 0.15)
        )
    ]
    prices = [candidate["valor_reais"] for candidate in plausible]
    return {
        "status_fipe": "enriquecido",
        "confianca_fipe": confidence(best["score"], margin),
        "fipe_categoria": category,
        "fipe_score_match": best["score"],
        "fipe_margem_segundo_candidato": margin,
        "fipe_quantidade_candidatos_plausiveis": len(plausible),
        "fipe_marca_sugerida": best["nome_marca"],
        "fipe_modelo_sugerido": best["nome_modelo"],
        "fipe_ano_modelo_sugerido": best["ano_modelo"],
        "fipe_combustivel_sugerido": best["nome_combustivel"],
        "fipe_codigo_sugerido": best["codigo_fipe"],
        "fipe_valor_sugerido": best["valor_reais"],
        "fipe_valor_min": min(prices),
        "fipe_valor_max": max(prices),
        "fipe_valor_mediano": round(statistics.median(prices)),
        "fipe_referencia": reference,
        "fipe_metodo_match": "marca_modelo_ano_aproximado_snapshot_local",
        "fipe_fonte": FIPEX_SOURCE_URL,
        "observacao_fipe": (
            "Valor aproximado por marca, modelo textual e ano; nao identifica o veiculo individual."
        ),
    }


def token_overlap(source: str, target: str) -> float:
    source_tokens = set(normalize_text(source).split())
    target_tokens = set(normalize_text(target).split())
    if not source_tokens:
        return 0.0
    return len(source_tokens & target_tokens) / len(source_tokens)


def enrich_dataset(
    input_path: Path,
    snapshot_path: Path,
    mapping_output: Path,
    vehicles_output: Path,
    output_path: Path,
) -> None:
    source_index, reference = load_snapshot(snapshot_path)
    with input_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=";")
        input_fields = reader.fieldnames or []
        detailed_rows = list(reader)

    representative_by_signature: dict[str, dict[str, str]] = {}
    vehicle_ids_by_signature: dict[str, set[str]] = defaultdict(set)
    representative_by_vehicle: dict[str, dict[str, str]] = {}
    for row in detailed_rows:
        signature = vehicle_signature(row)
        representative_by_signature.setdefault(signature, row)
        vehicle_id = vehicle_key(row)
        if vehicle_id not in {"", "NA"}:
            vehicle_ids_by_signature[signature].add(vehicle_id)
            representative_by_vehicle.setdefault(vehicle_id, row)

    mappings_by_signature = {}
    mapping_rows = []
    for position, (signature, row) in enumerate(
        representative_by_signature.items(), start=1
    ):
        if position % 1000 == 0:
            print(f"Processadas {position}/{len(representative_by_signature)} assinaturas")
        enrichment = match_vehicle(row, source_index, reference)
        brand, model = split_brand_model(row.get("marca", ""))
        mapping = {
            "assinatura_fipe": signature,
            "tipo_veiculo": row.get("tipo_veiculo", ""),
            "marca_modelo_original": row.get("marca", ""),
            "ano_fabricacao_veiculo": row.get("ano_fabricacao_veiculo", ""),
            "marca_normalizada_fipe": normalize_brand(brand),
            "modelo_normalizado_fipe": normalize_text(model),
            "quantidade_veiculos": len(vehicle_ids_by_signature[signature]),
            **enrichment,
        }
        mappings_by_signature[signature] = enrichment
        mapping_rows.append(mapping)

    mapping_rows.sort(
        key=lambda row: (
            -row["quantidade_veiculos"],
            row["tipo_veiculo"],
            row["marca_modelo_original"],
            row["ano_fabricacao_veiculo"],
        )
    )
    write_csv(mapping_output, mapping_rows, MAPPING_FIELDS)

    vehicle_rows = []
    for vehicle_id, row in representative_by_vehicle.items():
        vehicle_rows.append(
            {
                "id_veiculo": vehicle_id,
                "tipo_veiculo": row.get("tipo_veiculo", ""),
                "marca_modelo_original": row.get("marca", ""),
                "ano_fabricacao_veiculo": row.get("ano_fabricacao_veiculo", ""),
                **mappings_by_signature[vehicle_signature(row)],
            }
        )
    vehicle_rows.sort(key=lambda row: sort_vehicle_id(row["id_veiculo"]))
    write_csv(vehicles_output, vehicle_rows, VEHICLE_FIELDS)

    enriched_rows = [
        {**row, **mappings_by_signature[vehicle_signature(row)]}
        for row in detailed_rows
    ]
    write_csv(output_path, enriched_rows, input_fields + ENRICHMENT_FIELDS, delimiter=";")

    print(f"Referencia FIPE: {reference}")
    print(f"Assinaturas processadas: {len(mapping_rows)} em {mapping_output}")
    print(f"Veiculos unicos enriquecidos: {len(vehicle_rows)} em {vehicles_output}")
    print(f"Registros detalhados enriquecidos: {len(enriched_rows)} em {output_path}")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--snapshot", type=Path, required=True)
    parser.add_argument("--mapping-output", type=Path, required=True)
    parser.add_argument("--vehicles-output", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    enrich_dataset(
        input_path=args.input,
        snapshot_path=args.snapshot,
        mapping_output=args.mapping_output,
        vehicles_output=args.vehicles_output,
        output_path=args.output,
    )


''', 'enrich_fipe_local.py', 'exec'), namespace_enrich_fipe_local)
modulos['enrich_fipe_local'] = namespace_enrich_fipe_local
print('Código incorporado: enrich_fipe_local.py')


### filter_carros_latin_ncap.py

Mantem veiculos leves cujo pareamento Latin NCAP foi concluido e preserva a proveniencia anual.


In [ ]:
modulo_filter_carros_latin_ncap = types.ModuleType('notebook_filter_carros_latin_ncap')
sys.modules['notebook_filter_carros_latin_ncap'] = modulo_filter_carros_latin_ncap
namespace_filter_carros_latin_ncap = modulo_filter_carros_latin_ncap.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
from pathlib import Path

from enrich_fipe import normalize_text


CAR_LIKE_TYPES = {"AUTOMOVEL", "CAMINHONETE", "CAMIONETA", "UTILITARIO"}
PROVENANCE_FIELDS = ["ano_base", "arquivo_origem_enriquecido"]


def is_selected(row: dict[str, str]) -> bool:
    return (
        normalize_text(row.get("tipo_veiculo", "")) in CAR_LIKE_TYPES
        and row.get("status_latin_ncap", "").strip() == "enriquecido"
    )


def filter_dataset(
    input_path: Path,
    output_path: Path,
    ano_base: str,
    arquivo_origem_enriquecido: str,
) -> dict[str, int]:
    counts = {"lidos": 0, "selecionados": 0}
    with input_path.open("r", encoding="utf-8", newline="") as input_handle:
        reader = csv.DictReader(input_handle, delimiter=";")
        input_fields = reader.fieldnames or []
        missing_fields = {"tipo_veiculo", "status_latin_ncap"} - set(input_fields)
        if missing_fields:
            fields = ", ".join(sorted(missing_fields))
            raise ValueError(f"Colunas obrigatorias ausentes em {input_path}: {fields}")

        fieldnames = PROVENANCE_FIELDS + [
            field for field in input_fields if field not in PROVENANCE_FIELDS
        ]
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding="utf-8", newline="") as output_handle:
            writer = csv.DictWriter(output_handle, fieldnames=fieldnames, delimiter=";")
            writer.writeheader()
            for row in reader:
                counts["lidos"] += 1
                if not is_selected(row):
                    continue
                writer.writerow(
                    {
                        "ano_base": ano_base,
                        "arquivo_origem_enriquecido": arquivo_origem_enriquecido,
                        **row,
                    }
                )
                counts["selecionados"] += 1
    return counts


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--ano-base", required=True)
    parser.add_argument("--arquivo-origem-enriquecido", required=True)
    return parser


def main() -> None:
    args = build_parser().parse_args()
    counts = filter_dataset(
        input_path=args.input,
        output_path=args.output,
        ano_base=args.ano_base,
        arquivo_origem_enriquecido=args.arquivo_origem_enriquecido,
    )
    print(
        f"Registros Latin NCAP selecionados: {counts['selecionados']} "
        f"de {counts['lidos']} em {args.output}"
    )


''', 'filter_carros_latin_ncap.py', 'exec'), namespace_filter_carros_latin_ncap)
modulos['filter_carros_latin_ncap'] = namespace_filter_carros_latin_ncap
print('Código incorporado: filter_carros_latin_ncap.py')


### run_prf_multiyear_enrichment.py

Define a execução original ano a ano e as regras de retomada do processamento.

In [ ]:
modulo_run_prf_multiyear_enrichment = types.ModuleType('notebook_run_prf_multiyear_enrichment')
sys.modules['notebook_run_prf_multiyear_enrichment'] = modulo_run_prf_multiyear_enrichment
namespace_run_prf_multiyear_enrichment = modulo_run_prf_multiyear_enrichment.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

from build_prf_multiyear import combine
from enrich_fipe_local import FIPEX_SOURCE_URL


USER_AGENT = "prf-analise-multiyear-enrichment/1.0"


def year_csv(root: Path, year: int) -> Path:
    year_dir = root / str(year)
    paths = sorted(year_dir.glob("*.csv"))
    if len(paths) != 1:
        raise FileNotFoundError(f"Esperado 1 CSV em {year_dir}, encontrado {len(paths)}")
    return paths[0]


def run(command: list[str], resume_output: Path | None = None) -> None:
    if resume_output and resume_output.exists() and resume_output.stat().st_size > 0:
        print(f"Pulando etapa existente: {resume_output}", flush=True)
        return
    print("Executando:", " ".join(command), flush=True)
    subprocess.run(command, check=True)


def ensure_fipe_snapshot(snapshot_path: Path) -> None:
    if snapshot_path.exists() and snapshot_path.stat().st_size > 0:
        return
    snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        FIPEX_SOURCE_URL, headers={"User-Agent": USER_AGENT}
    )
    temporary_path = snapshot_path.with_suffix(snapshot_path.suffix + ".part")
    with urllib.request.urlopen(request, timeout=180) as response:
        with temporary_path.open("wb") as handle:
            shutil.copyfileobj(response, handle)
    temporary_path.replace(snapshot_path)


def enrich_year(
    year: int,
    pessoa_root: Path,
    ocorrencia_root: Path,
    output_root: Path,
    final_by_year_root: Path,
    filtered_by_year_root: Path,
    fipe_snapshot: Path,
    resume: bool,
) -> Path:
    pessoa_csv = year_csv(pessoa_root, year)
    ocorrencia_csv = year_csv(ocorrencia_root, year)
    year_output = output_root / str(year)
    final_year_output = final_by_year_root / str(year)
    filtered_year_output = filtered_by_year_root / str(year)
    year_output.mkdir(parents=True, exist_ok=True)
    final_year_output.mkdir(parents=True, exist_ok=True)
    filtered_year_output.mkdir(parents=True, exist_ok=True)

    family_output = year_output / "acidentes_com_familia_modelo.csv"
    latin_output = year_output / "acidentes_com_familia_latin_ncap.csv"
    final_output = final_year_output / "acidentes_enriquecido.csv"
    filtered_output = filtered_year_output / "acidentes_carros_latin_ncap.csv"

    resume_family = family_output if resume else None
    resume_latin = latin_output if resume else None
    resume_fipe = final_output if resume else None
    resume_filter = filtered_output if resume else None

    run(
        [
            sys.executable,
            "scripts/enrich_family_model.py",
            "--input",
            str(pessoa_csv),
            "--accidents",
            str(ocorrencia_csv),
            "--vehicles-output",
            str(year_output / "veiculos_com_familia_modelo.csv"),
            "--mapping-output",
            str(year_output / "mapeamento_familia_modelo.csv"),
            "--detailed-output",
            str(family_output),
            "--ranking-output",
            str(year_output / "ranking_familias_acidentes_fatais.csv"),
        ],
        resume_output=resume_family,
    )
    run(
        [
            sys.executable,
            "scripts/enrich_latin_ncap.py",
            "--input",
            str(family_output),
            "--cache",
            ".cache/latinncap/results.json",
            "--source-output",
            str(year_output / "latin_ncap_resultados.csv"),
            "--mapping-output",
            str(year_output / "mapeamento_familia_latin_ncap.csv"),
            "--output",
            str(latin_output),
        ],
        resume_output=resume_latin,
    )
    run(
        [
            sys.executable,
            "scripts/enrich_fipe_local.py",
            "--input",
            str(latin_output),
            "--snapshot",
            str(fipe_snapshot),
            "--mapping-output",
            str(year_output / "mapeamento_fipe_completo.csv"),
            "--vehicles-output",
            str(year_output / "veiculos_com_fipe.csv"),
            "--output",
            str(final_output),
        ],
        resume_output=resume_fipe,
    )
    run(
        [
            sys.executable,
            "scripts/filter_carros_latin_ncap.py",
            "--input",
            str(final_output),
            "--output",
            str(filtered_output),
            "--ano-base",
            str(year),
            "--arquivo-origem-enriquecido",
            str(Path(Path.cwd().name) / final_output),
        ],
        resume_output=resume_filter,
    )
    return final_output


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--start-year", type=int, default=2010)
    parser.add_argument("--end-year", type=int, default=2026)
    parser.add_argument(
        "--pessoa-root", type=Path, default=Path("data/raw/prf/acidentes/pessoa")
    )
    parser.add_argument(
        "--ocorrencia-root", type=Path, default=Path("data/raw/prf/acidentes/ocorrencia")
    )
    parser.add_argument(
        "--output-root",
        type=Path,
        default=Path("data/processed/multiyear/enrichment_by_year"),
    )
    parser.add_argument(
        "--final-by-year-root",
        type=Path,
        default=Path("data/processed/multiyear/final_by_year"),
    )
    parser.add_argument(
        "--filtered-by-year-root",
        type=Path,
        default=Path("data/processed/multiyear/carros_latin_ncap_by_year"),
    )
    parser.add_argument(
        "--latin-ncap-cars-output",
        type=Path,
        default=Path(
            "data/processed/multiyear/acidentes_2010_2026_carros_latin_ncap.csv"
        ),
    )
    parser.add_argument(
        "--fipe-snapshot",
        type=Path,
        default=Path("data/external/fipex/fipex-prices-2026-05.csv"),
    )
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--skip-final-combine", action="store_true")
    return parser


def main() -> None:
    args = build_parser().parse_args()
    ensure_fipe_snapshot(args.fipe_snapshot)
    for year in range(args.start_year, args.end_year + 1):
        print(f"\n=== Ano {year} ===", flush=True)
        enrich_year(
            year=year,
            pessoa_root=args.pessoa_root,
            ocorrencia_root=args.ocorrencia_root,
            output_root=args.output_root,
            final_by_year_root=args.final_by_year_root,
            filtered_by_year_root=args.filtered_by_year_root,
            fipe_snapshot=args.fipe_snapshot,
            resume=args.resume,
        )
    if not args.skip_final_combine:
        filtered_rows = combine(
            args.filtered_by_year_root,
            args.latin_ncap_cars_output,
            source_extra_fields=[],
        )
        print(
            "Registros finais de veiculos leves com Latin NCAP: "
            f"{filtered_rows} em {args.latin_ncap_cars_output}"
        )


''', 'run_prf_multiyear_enrichment.py', 'exec'), namespace_run_prf_multiyear_enrichment)
modulos['run_prf_multiyear_enrichment'] = namespace_run_prf_multiyear_enrichment
print('Código incorporado: run_prf_multiyear_enrichment.py')


### summarize_prf_multiyear.py

Calcula a cobertura anual e total dos enriquecimentos.

In [ ]:
modulo_summarize_prf_multiyear = types.ModuleType('notebook_summarize_prf_multiyear')
sys.modules['notebook_summarize_prf_multiyear'] = modulo_summarize_prf_multiyear
namespace_summarize_prf_multiyear = modulo_summarize_prf_multiyear.__dict__
exec(compile(r'''
#!/usr/bin/env python3

from __future__ import annotations

import argparse
import csv
from pathlib import Path
from typing import Any

from enrich_fipe import normalize_text


CAR_LIKE_TYPES = {"AUTOMOVEL", "CAMINHONETE", "CAMIONETA", "UTILITARIO"}
FIELDS = [
    "ano",
    "registros_detalhados",
    "veiculos",
    "veiculos_com_fipe",
    "cobertura_fipe_pct",
    "carros",
    "carros_com_fipe",
    "cobertura_fipe_carros_pct",
    "carros_com_latin_ncap",
    "cobertura_latin_ncap_carros_pct",
    "carros_em_acidentes_fatais",
    "carros_fatais_com_fipe",
    "cobertura_fipe_carros_fatais_pct",
    "carros_fatais_com_latin_ncap",
    "cobertura_latin_ncap_carros_fatais_pct",
]


def percentage(part: int, total: int) -> str:
    return f"{100 * part / total:.2f}" if total else ""


def summarize_year(year: str, path: Path) -> dict[str, Any]:
    vehicles: dict[str, dict[str, Any]] = {}
    detailed_rows = 0
    with path.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle, delimiter=";"):
            detailed_rows += 1
            vehicle_id = row.get("id_veiculo", "")
            if vehicle_id in {"", "NA"}:
                continue
            vehicle = vehicles.setdefault(
                vehicle_id,
                {
                    "carro": normalize_text(row.get("tipo_veiculo", ""))
                    in CAR_LIKE_TYPES,
                    "fipe": row.get("status_fipe") == "enriquecido",
                    "latin_ncap": row.get("status_latin_ncap") == "enriquecido",
                    "fatal": False,
                },
            )
            if row.get("classificacao_acidente", "").strip() == "Com Vítimas Fatais":
                vehicle["fatal"] = True

    cars = [vehicle for vehicle in vehicles.values() if vehicle["carro"]]
    fatal_cars = [vehicle for vehicle in cars if vehicle["fatal"]]
    vehicles_with_fipe = sum(vehicle["fipe"] for vehicle in vehicles.values())
    cars_with_fipe = sum(vehicle["fipe"] for vehicle in cars)
    cars_with_latin_ncap = sum(vehicle["latin_ncap"] for vehicle in cars)
    fatal_cars_with_fipe = sum(vehicle["fipe"] for vehicle in fatal_cars)
    fatal_cars_with_latin_ncap = sum(vehicle["latin_ncap"] for vehicle in fatal_cars)
    return {
        "ano": year,
        "registros_detalhados": detailed_rows,
        "veiculos": len(vehicles),
        "veiculos_com_fipe": vehicles_with_fipe,
        "cobertura_fipe_pct": percentage(vehicles_with_fipe, len(vehicles)),
        "carros": len(cars),
        "carros_com_fipe": cars_with_fipe,
        "cobertura_fipe_carros_pct": percentage(cars_with_fipe, len(cars)),
        "carros_com_latin_ncap": cars_with_latin_ncap,
        "cobertura_latin_ncap_carros_pct": percentage(cars_with_latin_ncap, len(cars)),
        "carros_em_acidentes_fatais": len(fatal_cars),
        "carros_fatais_com_fipe": fatal_cars_with_fipe,
        "cobertura_fipe_carros_fatais_pct": percentage(
            fatal_cars_with_fipe, len(fatal_cars)
        ),
        "carros_fatais_com_latin_ncap": fatal_cars_with_latin_ncap,
        "cobertura_latin_ncap_carros_fatais_pct": percentage(
            fatal_cars_with_latin_ncap, len(fatal_cars)
        ),
    }


def total_row(rows: list[dict[str, Any]]) -> dict[str, Any]:
    totals = {"ano": "TOTAL"}
    for field in FIELDS:
        if field == "ano" or field.endswith("_pct"):
            continue
        totals[field] = sum(int(row[field]) for row in rows)
    totals["cobertura_fipe_pct"] = percentage(
        totals["veiculos_com_fipe"], totals["veiculos"]
    )
    totals["cobertura_fipe_carros_pct"] = percentage(
        totals["carros_com_fipe"], totals["carros"]
    )
    totals["cobertura_latin_ncap_carros_pct"] = percentage(
        totals["carros_com_latin_ncap"], totals["carros"]
    )
    totals["cobertura_fipe_carros_fatais_pct"] = percentage(
        totals["carros_fatais_com_fipe"], totals["carros_em_acidentes_fatais"]
    )
    totals["cobertura_latin_ncap_carros_fatais_pct"] = percentage(
        totals["carros_fatais_com_latin_ncap"], totals["carros_em_acidentes_fatais"]
    )
    return totals


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--input-root",
        type=Path,
        default=Path("data/processed/multiyear/final_by_year"),
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/processed/multiyear/resumo_cobertura_2010_2026.csv"),
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()
    paths = sorted(args.input_root.glob("*/acidentes_enriquecido.csv"))
    rows = [summarize_year(path.parent.name, path) for path in paths]
    rows.append(total_row(rows))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    with args.output.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Resumo de cobertura: {args.output}")


''', 'summarize_prf_multiyear.py', 'exec'), namespace_summarize_prf_multiyear)
modulos['summarize_prf_multiyear'] = namespace_summarize_prf_multiyear
print('Código incorporado: summarize_prf_multiyear.py')
